# 01 - Data Quality Assessment
---

## Dataset - Expected Schema

| Field                   | Type    | Description                                     |
| ----------------------- | ------- | ----------------------------------------------- |
| `_id`                   | `str`   | Application ID (e.g., app_001)                  |
| `full_name`             | `str`   | Applicant full name                             |
| `email`                 | `str`   | Email address                                   |
| `ssn`                   | `str`   | Social Security Number - `NNN-NN-NNNN`          |
| `ip_address`            | `str`   | IP address at time of application               |
| `gender`                | `str`   | Gender - `Male` or `Female`                     |
| `date_of_birth`         | `date`  | Date of birth (`YYYY-MM-DD`)                    |
| `zip_code`              | `str`   | ZIP/postal code                                 |
| `annual_income`         | `float` | Annual income `> 0`                             |
| `credit_history_months` | `int`   | Months of credit history `≥ 0`                  |
| `debt_to_income`        | `float` | Debt-to-income ratio `0 - 1.0`                  |
| `savings_balance`       | `float` | Current savings balance `≥ 0`                   |
| `spend_<Category>`      | `float` | Spending by category `≥ 0` (pivot **spending_behavior**)|
| `loan_approved`         | `bool`  | `TRUE` or `FALSE`                               |
| `rejection_reason`      | `str`   | Reason (if denied)                              |
| `interest_rate`         | `float` | Assigned rate (if approved)                     |
| `approved_amount`       | `float` | Loan amount (if approved)                       |


---
## Phase 0 - Setup

In [87]:
import json
import re
import warnings
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
sns.set_theme(style="whitegrid", palette="muted")

print("✅ Environment ready")


✅ Environment ready


---

## Phase 1 - Loading & Normalization

**Goal:** Transform the nested JSON into a tabular DataFrame. Spending categories become individual columns (`spend_<Category>`) for feature engineering.


In [88]:
# 1‑A  Load JSON
with open("../data/raw_credit_applications.json", "r") as f:
    raw_data = json.load(f)

print(f"Records loaded: {len(raw_data):,}")
print(f"Top‑level keys (first record): {list(raw_data[0].keys())}")

Records loaded: 502
Top‑level keys (first record): ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp']


In [89]:
# 1‑B  Flatten nested dicts
df_raw = pd.json_normalize(
    raw_data,
    sep="_",                          # applicant_info_full_name, etc.
    errors="ignore",
)

# Rename flattened columns to cleaner names
rename_map = {c: c.replace("applicant_info_", "").replace("financials_", "").replace("decision_", "")
              for c in df_raw.columns}
df_raw.rename(columns=rename_map, inplace=True)

print(f"Shape after flatten: {df_raw.shape}")
df_raw.head(3)

Shape after flatten: (502, 21)


,_id,spending_behavior,processing_timestamp,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,rejection_reason,loan_purpose,interest_rate,approved_amount,annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


In [90]:
# 1‑C  Pivot spending_behavior into spend_<Category> columns
def pivot_spending(row):
    """Convert spending_behavior (list of dicts) into dicts keyed by category."""
    result = {}
    if isinstance(row, list):
        for item in row:
            cat = item.get("category", "Unknown")
            amt = item.get("amount", 0)
            result[f"spend_{cat.replace(' ', '_')}"] = amt
    return pd.Series(result)

spend_df = df_raw["spending_behavior"].apply(pivot_spending)
df = pd.concat([df_raw.drop(columns=["spending_behavior"]), spend_df], axis=1)

# Fill NaN spend columns with 0 (means didn't spend in that category)
spend_cols = [c for c in df.columns if c.startswith("spend_")]
df[spend_cols] = df[spend_cols].fillna(0).astype(float)

print(f"Shape after pivoting spending: {df.shape}")
print(f"Spending columns created: {spend_cols}")
df.head(3)

Shape after pivoting spending: (502, 35)
Spending columns created: ['spend_Shopping', 'spend_Rent', 'spend_Alcohol', 'spend_Dining', 'spend_Healthcare', 'spend_Fitness', 'spend_Entertainment', 'spend_Insurance', 'spend_Travel', 'spend_Transportation', 'spend_Utilities', 'spend_Groceries', 'spend_Education', 'spend_Adult_Entertainment', 'spend_Gambling']


,_id,processing_timestamp,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,rejection_reason,loan_purpose,interest_rate,approved_amount,annual_salary,notes,spend_Shopping,spend_Rent,spend_Alcohol,spend_Dining,spend_Healthcare,spend_Fitness,spend_Entertainment,spend_Insurance,spend_Travel,spend_Transportation,spend_Utilities,spend_Groceries,spend_Education,spend_Adult_Entertainment,spend_Gambling
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,480.0,790.0,247.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0.0,608.0,0.0,96.0,243.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0.0,109.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## Phase 2 - Data Quality Audit

Following **Detect -> Quantify -> Fix** cycle. 
---

### Dimension 1 - Accuracy

In [91]:
# 2.1  DETECT Duplicate _id values
dup_ids = df[df.duplicated(subset="_id", keep=False)].sort_values("_id")
print(f"🔍 Duplicate _id records found: {len(dup_ids)}")
dup_ids[["_id", "full_name", "loan_approved", "rejection_reason", "notes"]]

🔍 Duplicate _id records found: 4


,_id,full_name,loan_approved,rejection_reason,notes
383,app_001,Stephanie Nguyen,False,high_dti_ratio,NaN
455,app_001,Stephanie Nguyen,False,high_dti_ratio,DUPLICATE_ENTRY_ERROR
8,app_042,Joseph Lopez,False,algorithm_risk_score,NaN
354,app_042,Joseph Lopez,False,algorithm_risk_score,RESUBMISSION


In [92]:
# 2.1  QUANTIFY
print(f"Total records before dedup: {len(df)}")
print(f"Unique _ids:               {df['_id'].nunique()}")
print(f"Duplicated rows:           {df.duplicated(subset='_id').sum()}")
print()

# Show the notes field
for _, row in dup_ids.iterrows():
    note = row.get("notes", "—")
    print(f"  {row['_id']}  notes={note}")

Total records before dedup: 502
Unique _ids:               500
Duplicated rows:           2

  app_001  notes=nan
  app_001  notes=DUPLICATE_ENTRY_ERROR
  app_042  notes=nan
  app_042  notes=RESUBMISSION


In [93]:
# 2.1  FIX - Drop duplicates using the notes
#   DUPLICATE_ENTRY_ERROR  -> discard (data‑entry mistake)
#   RESUBMISSION           -> keep the resubmission (latest intent)
#   No notes               -> keep the first occurrence

df = df[df["notes"] != "DUPLICATE_ENTRY_ERROR"].copy()
df = df.drop_duplicates(subset="_id", keep="last").reset_index(drop=True)  #dropping duplicates keeping last (drops dup_e_e)

# Drop the notes column
df.drop(columns=["notes"], inplace=True, errors="ignore")

print(f"✅ Records after dedup: {len(df)} (removed {502 - len(df)} duplicates)")

✅ Records after dedup: 500 (removed 2 duplicates)


---
### Dimension 2 — Consistency

In [94]:
# 2.2‑A  DETECT - Inconsistent gender coding
print("🔍 Gender value counts (before):")
print(df["gender"].value_counts(dropna=False))

🔍 Gender value counts (before):
gender
Male      194
Female    193
F          58
M          53
            2
Name: count, dtype: int64


In [95]:
# 2.2‑A  FIX - Standardise gender to Male / Female / Unknown
gender_map = {
    "M": "Male",
    "F": "Female",
    "Male": "Male",
    "Female": "Female",
}
df["gender"] = df["gender"].map(gender_map).fillna("Unknown")

print("✅ Gender value counts (after):")
print(df["gender"].value_counts())

✅ Gender value counts (after):
gender
Female     251
Male       247
Unknown      2
Name: count, dtype: int64


In [96]:
# 2.2‑B  DETECT - Income schema mistakes
#  Some records store income under 'annual_salary' instead of 'annual_income'
#  Some store annual_income as a string or float

if "annual_salary" in df.columns:
    salary_mask = df["annual_salary"].notna()
    print(f"🔍 Records using 'annual_salary' key: {salary_mask.sum()}")
else:
    salary_mask = pd.Series(False, index=df.index)
    print("🔍 No 'annual_salary' column found")

# Check types in annual_income
print(f"\n   annual_income dtype: {df['annual_income'].dtype}")
print("   Sample non‑numeric values:")
for idx, val in df["annual_income"].items():
    if isinstance(val, str):
        print(f"     Row {idx} (_id={df.loc[idx, '_id']}): '{val}' (str)")

🔍 Records using 'annual_salary' key: 5

   annual_income dtype: object
   Sample non‑numeric values:
     Row 91 (_id=app_088): '55000' (str)
     Row 145 (_id=app_135): '65000' (str)
     Row 170 (_id=app_446): '73000' (str)
     Row 305 (_id=app_389): '51000' (str)
     Row 333 (_id=app_026): '72000' (str)
     Row 430 (_id=app_312): '80000' (str)
     Row 446 (_id=app_180): '111000' (str)
     Row 488 (_id=app_224): '93000' (str)


In [97]:
# 2.2‑B  FIX - Merge annual_salary and annual_income, turn to numeric
if "annual_salary" in df.columns:
    # Fill missing annual_income with annual_salary when available
    df["annual_income"] = df["annual_income"].fillna(df["annual_salary"])
    df.drop(columns=["annual_salary"], inplace=True)

# To numeric (strings to float, invalid to NaN)
df["annual_income"] = pd.to_numeric(df["annual_income"], errors="coerce")

print(f"✅ annual_income dtype: {df['annual_income'].dtype}")
print(f"   Null count after coercion: {df['annual_income'].isna().sum()}")
print(f"   Range: {df['annual_income'].min():,.0f} – {df['annual_income'].max():,.0f}")

✅ annual_income dtype: float64
   Null count after coercion: 0
   Range: 0 – 171,000


In [98]:
# 2.2‑C  DETECT - Inconsistent date‑of‑birth formats
print("🔍 Sample date_of_birth values (raw):")
sample_dobs = df["date_of_birth"].dropna().unique()
np.random.seed(42)
for dob in np.random.choice(sample_dobs, size=min(12, len(sample_dobs)), replace=False):
    print(f"   {dob}")

# Quick regex classification
fmt_counts = {"YYYY-MM-DD": 0, "DD/MM/YYYY": 0, "YYYY/MM/DD": 0, "empty/null": 0, "other": 0}
for dob in df["date_of_birth"]:
    if pd.isna(dob) or str(dob).strip() == "":
        fmt_counts["empty/null"] += 1
    elif re.match(r"^\d{4}-\d{2}-\d{2}$", str(dob)):
        fmt_counts["YYYY-MM-DD"] += 1
    elif re.match(r"^\d{2}/\d{2}/\d{4}$", str(dob)):
        fmt_counts["DD/MM/YYYY"] += 1
    elif re.match(r"^\d{4}/\d{2}/\d{2}$", str(dob)):
        fmt_counts["YYYY/MM/DD"] += 1
    else:
        fmt_counts["other"] += 1

print(f"\n🔍 Format distribution: {fmt_counts}")

🔍 Sample date_of_birth values (raw):
   1998-07-21
   11/03/1967
   18/03/1988
   1987-10-23
   1992-09-01
   13/06/1993
   1975-04-14
   1985-09-06
   24/10/1988
   1970-10-01
   1999/06/16
   1983-10-26

🔍 Format distribution: {'YYYY-MM-DD': 339, 'DD/MM/YYYY': 101, 'YYYY/MM/DD': 56, 'empty/null': 4, 'other': 0}


In [99]:
# 2.2‑C  FIX - Parse all dates to ISO 8601 and derive age

def parse_dob(dob_str):
    """Attempt multiple date formats and return a datetime or NaT."""
    if pd.isna(dob_str) or str(dob_str).strip() == "":
        return pd.NaT

    dob_str = str(dob_str).strip()

    # Try YYYY-MM-DD and YYYY/MM/DD
    for fmt in ("%Y-%m-%d", "%Y/%m/%d"):
        try:
            return datetime.strptime(dob_str, fmt)
        except ValueError:
            pass

    # DD/MM/YYYY (day‑first)
    if re.match(r"^\d{2}/\d{2}/\d{4}$", dob_str):
        try:
            return datetime.strptime(dob_str, "%d/%m/%Y")
        except ValueError:
            pass

    return pd.NaT

df["date_of_birth"] = df["date_of_birth"].apply(parse_dob)
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"])

# Derive age (in 2024‑01‑15, the processing date in the data)
reference_date = pd.Timestamp("2024-01-15")
df["age"] = ((reference_date - df["date_of_birth"]).dt.days / 365.25).apply(
    lambda x: int(round(x)) if pd.notna(x) else pd.NA
).astype("Int64")

print(f"✅ date_of_birth dtype: {df['date_of_birth'].dtype}")
print(f"   Null DOBs: {df['date_of_birth'].isna().sum()}")
print(f"   Age range: {df['age'].min()} – {df['age'].max()}")
df[["_id", "date_of_birth", "age"]].head(8)


✅ date_of_birth dtype: datetime64[ns]
   Null DOBs: 30
   Age range: 22 – 65


,_id,date_of_birth,age
0,app_200,2001-03-09,23
1,app_037,1992-03-31,32
2,app_215,1989-10-24,34
3,app_024,1983-04-25,41
4,app_184,1999-05-21,25
5,app_275,1982-02-14,42
6,app_099,1990-01-28,34
7,app_246,1991-10-11,32


---
### Dimension 3 — Validity